# T8 — head Bi-GRU/Bi-LSTM/CNN đọc token PhoBERT, chạy trên GPU Colab

PhoBERT vẫn **đóng băng**. Trên máy chính, một epoch của head này đo được
**23 phút (CPU)** / **8,5 phút (MPS)** — quá chậm cho `patience=8`. Notebook này chạy
đúng cùng mã nguồn trên GPU Colab, nơi `nn.GRU`/`nn.LSTM` dùng kernel cuDNN.

## Không cần upload gì

Cache token-level nặng 28 GB, nhưng nó là thứ **sinh ra được** chứ không phải dữ
liệu gốc: upload mất 1,5–3 giờ, còn encode lại trên GPU chỉ vài phút. Notebook vì
vậy tự dựng lại toàn bộ từ nguồn:

1. Clone nhánh `t8-rnn-colab`.
2. Tải `VietJobs.csv` từ HuggingFace, **xác nhận sha256**, `dataset build` lại splits
   — tất định cùng `SPLIT_SEED`, nên ra đúng 34.354 / 3.812 / 9.541 dòng như máy chính.
3. `encode --pooling none` cho hai họ cột `raw` (phân loại) và `masked` (lương).
4. Chạy `pytest` canh cache, rồi 5 lần huấn luyện `--device cuda`.
5. Đóng gói kết quả (vài MB) để mang về máy chính.

**Chỉ cần**: Runtime → Change runtime type → **T4 GPU** trước khi bấm chạy.

In [ ]:
!nvidia-smi

## 1. Clone mã nguồn

In [ ]:
!git clone -b t8-rnn-colab https://github.com/boy2407/vietjobs.git
%cd vietjobs
!git log --oneline -3

## 2. Cài phụ thuộc

**Không** cài lại `torch` — bản CUDA sẵn có của Colab phải giữ nguyên;
`requirements-dl.txt` tự ghi rõ điều này ("Trên Colab/Kaggle thì bỏ ghim torch").

In [ ]:
!pip install -q "transformers>=4.38,<4.47" "pyarrow>=15.0" "scikit-learn>=1.5" \
    "scipy>=1.13" "joblib>=1.4" "underthesea>=6.8" huggingface_hub pytest
!pip install -q -e . --no-deps
%env PYTHONPATH=src

## 3. Dữ liệu gốc + dựng lại splits

`data/README.md` cam kết splits byte-với-byte giống nhau trên mọi máy khi cùng
`SPLIT_SEED`. Bước dưới xác nhận sha256 **trước** khi build — sai tệp gốc thì mọi
thứ sau đó sai theo mà không có cảnh báo nào.

In [ ]:
import pathlib, shutil, hashlib
from huggingface_hub import hf_hub_download

EXPECTED_SHA256 = "85862b06fda4e814fe0c1d8622f173d189c92758345f77df16d1232d0c49d477"

p = hf_hub_download("dinhieufam/VietJobs", "VietJobs.csv", repo_type="dataset",
                    cache_dir="data/raw/.cache/huggingface")
pathlib.Path("data/raw").mkdir(parents=True, exist_ok=True)
shutil.copy(p, "data/raw/VietJobs.csv")

h = hashlib.sha256(pathlib.Path("data/raw/VietJobs.csv").read_bytes()).hexdigest()
assert h == EXPECTED_SHA256, f"sha256 lệch: {h} != {EXPECTED_SHA256} — DỪNG, đừng build"
print("sha256 khớp:", h)

In [ ]:
# Tách từ 47.707 tin bằng underthesea — chạy trên CPU, ~13 phút, không dùng GPU.
!python -m vietjobs.dataset build

In [ ]:
import json
m = json.load(open("data/processed/manifest.json"))
assert m["splits"]["train"]["rows"] == 34354
assert m["splits"]["dev"]["rows"] == 3812
assert m["splits"]["test"]["rows"] == 9541
assert m["groups_straddling_splits"] == 0
print("manifest khớp máy chính:", m["splits"])

## 4. Nhúng PhoBERT — hai mức, trên GPU

`--pooling mean` cho cache gộp `[n, 768]` (dùng làm mốc so sánh và để test canh),
`--pooling none` cho cache token `[n, 256, 768]` float16 mà head RNN đọc.

Cỡ đĩa sinh ra: ~28 GB cho bốn tệp token (`train`/`dev` × `raw`/`masked`). Đĩa
Colab thường còn 70–100 GB nên đủ; cell dưới in ra dung lượng còn trống trước khi chạy.

In [ ]:
!df -h /content | tail -1

In [ ]:
# Bản gộp — nhanh, vài phút mỗi họ cột.
!python -m vietjobs.dl.encode --task category --splits train dev --device cuda
!python -m vietjobs.dl.encode --task salary   --splits train dev --device cuda

In [ ]:
# Bản token — cùng một lượt forward, chỉ khác chỗ ghi ra đĩa (13 GB mỗi tệp train).
!python -m vietjobs.dl.encode --task category --splits train dev --pooling none --device cuda
!python -m vietjobs.dl.encode --task salary   --splits train dev --pooling none --device cuda
!ls -la artifacts/embeddings/

## 5. Canh cache đúng trước khi huấn luyện

`test_dl_encode.py` so `mean(cache token)` với cache gộp — nếu hai thứ lệch nhau thì
cache hỏng, biết ngay thay vì để nhiều giờ GPU chạy trên dữ liệu sai.

In [ ]:
!python -m pytest -q tests/test_dl_encode.py tests/test_dl_heads.py

## 6. Năm lần chạy — T8.3, T8.4, T8.5

Mốc so sánh (đo trên máy chính, CPU): `dl-cat-ce-cpu-0920` macro-F1 **0,6030**,
`dl-cat-focal-cpu-0920` **0,5972**, `dl-sal-cpu-0920` MAE **4,13 tr**.

Ngưỡng cải thiện thật: Δ macro-F1 > **0,015** — nhiễu chạy lại cùng cấu hình đã đo
được là 0,0017 (ba lần `dl-cat-*`: 0,6013 · 0,6025 · 0,6030).

In [ ]:
# T8.3 — hai hàm mất mát, cùng seed mặc định
!python -m vietjobs.dl.train_dl --task category --head rnn --loss ce    --device cuda --run-id dl-cat-rnn-ce

In [ ]:
!python -m vietjobs.dl.train_dl --task category --head rnn --loss focal --device cuda --run-id dl-cat-rnn-focal

In [ ]:
# T8.4 — lương; family(task) tự chọn cache họ `masked` (quy tắc chống rò rỉ)
!python -m vietjobs.dl.train_dl --task salary --head rnn --device cuda --run-id dl-sal-rnn

In [ ]:
# T8.5 — ablation nhánh. "both" đã có ở dl-cat-rnn-ce, chỉ cần thêm gru/lstm riêng.
!python -m vietjobs.dl.train_dl --task category --head rnn --branches gru  --device cuda --run-id dl-cat-rnn-gru
!python -m vietjobs.dl.train_dl --task category --head rnn --branches lstm --device cuda --run-id dl-cat-rnn-lstm

## 7. Đóng gói kết quả mang về (vài MB, không có cache)

In [ ]:
import shutil, pathlib

RUN_IDS = ["dl-cat-rnn-ce", "dl-cat-rnn-focal", "dl-sal-rnn",
           "dl-cat-rnn-gru", "dl-cat-rnn-lstm"]

bundle = pathlib.Path("t8_results")
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
for r in RUN_IDS:
    src = pathlib.Path("artifacts") / r
    if src.exists():
        shutil.copytree(src, bundle / r)
    else:
        print("THIẾU:", r)
shutil.copy("docs/04-results.md", bundle / "04-results.md")

zip_path = shutil.make_archive("t8_results", "zip", bundle)
print("zip:", zip_path, pathlib.Path(zip_path).stat().st_size / 1e6, "MB")

from google.colab import files
files.download(zip_path)

## 8. Mang kết quả về máy chính

1. Giải nén `t8_results.zip`.
2. Chép 5 thư mục `dl-cat-rnn-*` / `dl-sal-rnn` vào `artifacts/` của repo cục bộ.
3. Đưa `04-results.md` trong zip cho tôi — `docs/04-results.md` ở máy chính là
   **chỉ-thêm**, tôi ghép đúng 5 dòng mới vào cuối, không sửa dòng cũ.
4. Tôi đọc số thật từ `metrics.json` / `history.jsonl` trong mỗi thư mục rồi viết
   kết luận vào `docs/06-baseline-dl.md`, bảng ablation T8.5 và chương 4 luận văn
   theo Rule 10 — không chép số bằng tay.

**Nếu phiên Colab rớt giữa chừng:** cache trong `artifacts/embeddings/` mất theo, phải
chạy lại từ cell 4. Muốn tránh, mount Drive rồi copy 4 tệp `*-tok.f16.npy` lên đó sau
khi encode xong — nhưng 28 GB lên Drive chậm hơn encode lại, nên thường không đáng.